# UniFashion — controlled learning-rate / batch-size tuning
Stop the original training process after **Saved resumable checkpoint ...**.
This notebook keeps its `last.pt` unchanged. It creates one fixed parent snapshot,
then warm-starts each trial independently with a new optimizer.

Upload `checkpoints/unifashion_tuning.zip` to the existing
`MyDrive/Thesis/UniFashion/week8/inputs`. Keep the original Week 8 code/data ZIPs there.
Use a GPU runtime with Python 3.10–3.12. Trial training and evaluation stay FP32.
This is a short continuation study, not a repeat of the full paper training.

Cells can be rerun after a quota interruption: keep the **same STUDY_NAME**.
Change it only when intentionally creating a new study/parent checkpoint.

In [1]:
from pathlib import Path
from google.colab import drive
import os, sys, json, zipfile, subprocess, shutil
from datetime import datetime, timezone
drive.mount('/content/drive')
WORK = Path('/content/unifashion_week8')
PERSIST = Path('/content/drive/MyDrive/Thesis/UniFashion/week8')
TRAIN_RUN = PERSIST / 'runs/full_train_fp32_b2_seed42'
STUDY_NAME = 'fp32_lr_batch_01'
EXPERIMENT = PERSIST / 'tuning' / STUDY_NAME
WORK.mkdir(parents=True, exist_ok=True)
for name in ('unifashion_week8_code.zip', 'unifashion_week8_data.zip', 'unifashion_tuning.zip'):
    if name == 'unifashion_week8_data.zip' and (WORK/'data/data_audit.json').exists() and (WORK/'images').exists():
        continue
    with zipfile.ZipFile(PERSIST/'inputs'/name) as archive:
        for member in archive.namelist():
            if not (WORK/member).resolve().is_relative_to(WORK.resolve()):
                raise ValueError(member)
        archive.extractall(WORK)
os.chdir(WORK)
os.environ['HF_HOME'] = str(WORK/'.cache/huggingface')
cache = PERSIST/'pretrained_assets'
cache.mkdir(parents=True, exist_ok=True)
(WORK/'checkpoints').mkdir(exist_ok=True)
link = WORK/'checkpoints/unifashion'
if not link.exists():
    link.symlink_to(cache, target_is_directory=True)

def run_live(command):
    process = subprocess.Popen(command, cwd=WORK, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError(f'Command failed ({process.returncode}); see output above')
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        raise


Mounted at /content/drive


## 1. Install, check GPU, and run CPU regression fixtures

In [2]:
if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError('Use a Python 3.10-3.12 runtime')
run_live([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-unifashion.txt'])
run_live([sys.executable, '-B', 'scripts/check_unifashion_environment.py'])
run_live([sys.executable, '-c', 'import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name())'])
run_live([sys.executable, '-B', '-u', 'scripts/download_unifashion.py', '--with-vision'])
run_live([sys.executable, '-B', '-m', 'pytest', 'tests/test_unifashion_tuning.py',
          'tests/test_fashioniq_evaluation.py', '-q'])


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 18.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 32.2 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 12.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB

## 2. Fix the parent snapshot and prepare shared inputs
First use copies the current saved checkpoint; progress is read from that file,
not assumed to be step 60,000. Later runs reuse that exact snapshot.
Allow roughly 10–12 GB extra Drive space for parent and three resumable trial checkpoints
(plus temporary checkpoint writes). No weights are put in the evidence ZIP.

In [3]:
if not (EXPERIMENT/'parent.json').exists():
    run_live([sys.executable, '-B', '-u', 'scripts/tune_unifashion.py', 'snapshot',
              '--source', str(TRAIN_RUN/'last.pt'), '--output', str(EXPERIMENT)])
PARENT = EXPERIMENT/'parent_snapshot.pt'
print((EXPERIMENT/'parent.json').read_text())
if not (EXPERIMENT/'inputs/study.json').exists():
    run_live([sys.executable, '-B', '-u', 'scripts/tune_unifashion.py', 'prepare',
              '--data-dir', str(WORK/'data'),
              '--eval-data', str(WORK/'evaluation_inputs/evaluation_data'),
              '--output', str(EXPERIMENT/'inputs')])
study = json.loads((EXPERIMENT/'inputs/study.json').read_text())
assert study['config'] == json.loads((WORK/'configs/unifashion_tuning.json').read_text()), 'Use a new study name after changing settings'
print('Train examples:', study['train_examples'], 'Screen queries:', study['screen_queries'])

def evaluate_checkpoint(checkpoint, name, stage):
    output = EXPERIMENT/stage/name
    if (output/'status.json').exists():
        status = json.loads((output/'status.json').read_text())
        if status.get('status') == 'evaluation_completed':
            print('Reusing completed evaluation:', output)
            return
    if output.exists():
        # Keep interrupted output for diagnosis. Evaluation itself restarts.
        archived = output.with_name(name + '_interrupted_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
        output.rename(archived)
    inputs = EXPERIMENT/'inputs/screen/evaluation_data' if stage == 'screen' else WORK/'evaluation_inputs/evaluation_data'
    command = [sys.executable, '-B', '-u', 'scripts/evaluate_unifashion_fashioniq.py',
               '--checkpoint', str(checkpoint), '--eval-data', str(inputs),
               '--data-dir', str(WORK/'data'), '--images-dir', str(WORK/'images'),
               '--output', str(output), '--batch-size', '4']
    if stage == 'screen':
        command.append('--allow-screening')
    run_live(command)


{
  "checkpoint_sha256": "df8226a0e61e41ebb5bf88f1870b0176118f75d14612ceb8c0e04cf91309a683",
  "progress": {
    "checkpoint_kind": "full_training",
    "optimizer_steps": 60000,
    "completed_epochs": 6,
    "next_epoch_one_based": 7,
    "completed_batches_in_current_epoch": 6000,
    "planned_epochs": 10
  },
  "config": {
    "seed": 42,
    "epochs": 10,
    "batch_size": 2,
    "learning_rate": 2e-05,
    "weight_decay": 0.05,
    "betas": [
      0.9,
      0.98
    ],
    "eps": 1e-07,
    "scheduler": "one_cycle",
    "warmup_epochs": 1.5,
    "div_factor": 100.0,
    "final_div_factor": 10000.0,
    "max_grad_norm": 1.0,
    "image_size": 224,
    "target_ratio": 1.25,
    "max_txt_len": 64,
    "num_workers": 0,
    "precision": "fp32",
    "grad_scaler_init_scale": 256.0,
    "max_amp_retries": 10,
    "freeze_vit": true,
    "checkpoint_every_steps": 500,
    "monitor_per_category": 32,
    "description": "Full FashionIQ train, upstream 10-epoch OneCycle/AdamW settings; F

## 3. Screen the unchanged parent
192 fixed validation queries (64/category), 512 gallery images/category.
These reduced-gallery scores are **only for screening**; do not compare them with
Week 9 or the published paper. The full-validation cell below uses all 6,016 queries.

In [4]:
evaluate_checkpoint(PARENT, 'base', 'screen')


Reading and hashing the checkpoint snapshot...
Verifying 1534 validation images...
Verifying the frozen EVA-G checkpoint...
Snapshot progress: {"optimizer_steps": 60000, "completed_epochs": 6, "next_epoch_one_based": 7, "completed_batches_in_current_epoch": 6000}
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
dress: gallery 100/512
dress: gallery 200/512
dress: gallery 300/512
dress: gallery 400/512
dress: gallery 500/512
dress: gallery 512/512
dress: queries 64/64
shirt: gallery 100/512
shirt: gallery 200/512
shirt: gallery 300/512
shirt:

## 4. Train and screen each trial
All trials use the same 1,200 examples once, the same order, FP32 and fresh AdamW:
`lr1e5_b2` (600 updates), `lr5e6_b2` (600), `lr1e5_b4` (300).
This controls example exposure, not update count; batch size also changes contrastive negatives.
Training saves every 100 updates and resumes from `trial.pt`.
If batch 4 runs out of memory, record the failure; do not silently change that trial's batch.

In [5]:
for trial in study['config']['trials']:
    name = trial['name']
    output = EXPERIMENT/'trials'/name
    status = json.loads((output/'status.json').read_text()) if (output/'status.json').exists() else {}
    if status.get('status') != 'trial_completed':
        if output.exists() and not (output/'trial.pt').exists():
            print('No resumable checkpoint; preserving failed folder and moving to next trial:', name)
            continue
        command = [sys.executable, '-B', '-u', 'scripts/tune_unifashion.py', 'train',
                   '--study', str(EXPERIMENT/'inputs'), '--parent', str(PARENT),
                   '--trial', name, '--data-dir', str(WORK/'data'),
                   '--images-dir', str(WORK/'images'), '--output', str(output)]
        if (output/'trial.pt').exists():
            command.append('--resume')
        try:
            run_live(command)
        except RuntimeError:
            print('Trial failed; details retained in status.json/run.log:', name)
            continue
    evaluate_checkpoint(output/'trial.pt', name, 'screen')
run_live([sys.executable, '-B', 'scripts/report_unifashion_tuning.py', '--root', str(EXPERIMENT)])
selection = json.loads((EXPERIMENT/'report/selection.json').read_text())
print(selection)


Verified 10000/40151 images
Verified 20000/40151 images
Verified 30000/40151 images
Verified 40000/40151 images
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Saved trial checkpoint: 0/600 updates
{"update": 1, "samples": 2, "lr": 1e-05, "loss_itc": 0.0, "loss_itm": 6.596205366804497e-06, "loss_ttc": 0.0, "loss": 6.596205366804497e-06, "grad_norm_before_clip": 0.00016740287537686527, "amp_retries": 0, "loss_scale_before": 1.0, "loss_scale_after": 1.0, "seconds": 2.1967733860001317}
{"update": 25, "samples": 2, "lr": 1e-05, "loss_itc": 0.0,

## 5. Inspect screening and confirm on full validation
Choose the largest mean of macro R@10 and R@50 with gallery-caption scoring.
Ties keep the parent; do not select by raw training loss across different batches.
Failed trials are reported and excluded. Partial/pending studies do not produce a winner.

Run this cell when time/quota permits: two full evaluations can take about two hours
on the prior T4 setup (previous full evaluation was about 62 minutes). Actual runtime varies.
If screening retains base, only base is evaluated. Already completed evaluations are reused.
Even if screening improves, full validation may not improve.

In [ ]:
from IPython.display import display, Markdown
display(Markdown((EXPERIMENT/'report/weekly_report.md').read_text()))
selection = json.loads((EXPERIMENT/'report/selection.json').read_text())
assert selection['ready'], 'Finish pending screening evaluations before full confirmation'
evaluate_checkpoint(PARENT, 'base', 'full')
if selection['selected'] != 'base':
    name = selection['selected']
    evaluate_checkpoint(EXPERIMENT/'trials'/name/'trial.pt', name, 'full')
run_live([sys.executable, '-B', 'scripts/report_unifashion_tuning.py', '--root', str(EXPERIMENT)])
display(Markdown((EXPERIMENT/'report/weekly_report.md').read_text()))


# Tuần 10 — Tinh chỉnh learning rate và batch size

Parent SHA-256: `df8226a0e61e41ebb5bf88f1870b0176118f75d14612ceb8c0e04cf91309a683`. Tiến độ thực tế: `{'checkpoint_kind': 'full_training', 'optimizer_steps': 60000, 'completed_epochs': 6, 'next_epoch_one_based': 7, 'completed_batches_in_current_epoch': 6000, 'planned_epochs': 10}`.

Mỗi trial khởi tạo lại AdamW từ cùng weights; FP32, LR cố định, cùng 1 lượt trên tập train con 1200 mẫu. Không tiếp tục optimizer/OneCycle của run gốc. Batch lớn hơn có ít update hơn nhưng cùng số ví dụ; số negative trong batch cũng thay đổi.

## Trạng thái trial

| Trial | LR | Batch | Trạng thái |
|---|---:|---:|---|
| lr1e5_b2 | 1e-05 | 2 | trial_completed |
| lr5e6_b2 | 5e-06 | 2 | trial_completed |
| lr1e5_b4 | 1e-05 | 4 | trial_completed |

## Screening — gallery giảm, không so với paper/Tuần 9

| Model | Macro R@10 (%) | Macro R@50 (%) | Mean R (%) |
|---|---:|---:|---:|
| base | 88.02 | 96.35 | 92.19 |
| lr1e5_b2 | 87.50 | 95.83 | 91.67 |
| lr5e6_b2 | 87.50 | 96.35 | 91.93 |
| lr1e5_b4 | 90.10 | 95.31 | 92.71 |

## Full validation — cùng giao thức Tuần 9

| Model | Macro R@10 (%) | Macro R@50 (%) | Mean R (%) |
|---|---:|---:|---:|
Chưa có kết quả; không điền số giả hoặc chọn hyperparameter tốt nhất.

Shortlist: **lr1e5_b4**. Tie chọn base; trial lỗi bị loại và ghi rõ, không tự giảm batch rồi giữ nguyên tên trial.

## Đối chiếu số liệu công bố

UniFashion, Table 4: macro R@10=58,93%, R@50=76,93%. [Paper EMNLP 2024](https://aclanthology.org/2024.emnlp-main.89.pdf). Chênh lệch chỉ là tham chiếu số học: khác caption gallery, text preprocessing, reference exclusion, precision và batch; chưa phải tái lập protocol paper.

Chưa có cặp full-val base/trial được chọn để kết luận cải thiện do tuning. Nếu shortlist là base thì chưa tìm được cấu hình tốt hơn ở vòng screening này.

Kết quả một seed và ngân sách ngắn chỉ mang tính thăm dò. Validation được dùng chọn cấu hình; không gọi là test độc lập. Muốn đánh giá ổn định cần thêm seed/ngân sách. Báo cáo cả scoring có/không caption trong JSON; tiêu chí chọn đã cố định là scoring có caption.


Reading and hashing the checkpoint snapshot...
Verifying 15415 validation images...
Verifying the frozen EVA-G checkpoint...
Snapshot progress: {"optimizer_steps": 60000, "completed_epochs": 6, "next_epoch_one_based": 7, "completed_batches_in_current_epoch": 6000}
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
dress: gallery 100/3817
dress: gallery 200/3817
dress: gallery 300/3817
dress: gallery 400/3817
dress: gallery 500/3817
dress: gallery 600/3817
dress: gallery 700/3817
dress: gallery 800/3817
dress: gallery 900/3817
dress: gallery 10

## 6. Export evidence (can run before full evaluation to report partial progress)
Send `tuning_evidence.zip` back for the final analysis/report.
The original full training run remains resumable with its original notebook and `last.pt`.
Trial checkpoints belong to this study and are not replacements for the original run.

In [ ]:
run_live([sys.executable, '-B', 'scripts/report_unifashion_tuning.py', '--root', str(EXPERIMENT)])
evidence = EXPERIMENT/'tuning_evidence.zip'
with zipfile.ZipFile(evidence, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(EXPERIMENT.rglob('*')):
        if path.is_file() and path.suffix.lower() in {'.json', '.csv', '.md', '.png', '.log', '.xml'}:
            if path.name.endswith('_partial.csv'):
                continue
            z.write(path, path.relative_to(EXPERIMENT))
print('Send:', evidence)
